# Insurance Policy Renewal — AutoML Data Preparation with PyCaret, solved

01 Core Python · 02 Pandas · 03 Cleaning · 04 Transformation · 05 Feature Engineering · 06 Regression · 07 Model Prep · **▶ 08 Case Studies**

`08_Case_Studies/04_insurance_policy_renewal_automl_pycaret.ipynb`

---

### In one paragraph (no jargon)

This notebook is the worked solution to the *AutoML Data Preparation with PyCaret* class exercise
(`reference_documents/automl-pycaret-student-task.pptx`). An insurance company wants to predict
whether an existing customer will **renew** their policy (`Policy_Renewed`: Yes/No). The dataset,
`datasets/insurance.csv`, is deliberately dirty — the same way the `FinServe` lending case in
`reference_documents/automl-data-preparation-case-finserve.docx` describes a "94% accuracy trap":
an AutoML tool can automate a lot of technical preprocessing, but it cannot tell the difference
between a genuine value and a data-entry mistake, and it cannot know that a field only exists
*after* the outcome is already known. We do the human/domain work first, then hand the rest —
imputation, encoding, scaling, outlier removal, class balancing, model comparison — to PyCaret.

## Learning objective

**AutoML can automate many technical preprocessing tasks, but it cannot reliably replace human/domain judgment.**

The dataset contains both:

1. **AutoML-manageable issues** — ordinary missing values, categorical encoding, scaling, class imbalance and statistical outliers.
2. **Human-required issues** — impossible values, sentinel codes, inconsistent business labels, exact duplicates, an identifier field, invalid/mixed dates and a post-outcome leakage variable.

**Target:** `Policy_Renewed` (`Yes` / `No`)

## What this dataset contains

| Data issue | Example in this dataset | Who should handle it? |
|---|---|---|
| Missing numeric values | Missing `Annual_Income`, `Annual_Premium`, `Age` | PyCaret can impute |
| Missing categorical values | Missing `Occupation`, `Region`, `Payment_Mode` | PyCaret can impute |
| Categorical encoding | `Policy_Type`, `Occupation`, `Region`, `Payment_Mode` | PyCaret can encode |
| Different numerical scales | Income (lakhs) vs. number of claims (single digits) | PyCaret can normalize |
| Statistical outliers | A very large `Total_Claim_Amount` | PyCaret can remove statistically |
| Class imbalance | Far more renewals than non-renewals | PyCaret can use SMOTE |
| Impossible values | `Age` = 145 or -4 | **Human/domain rule** |
| Sentinel values | `Annual_Income` = 9,999,999; `Annual_Premium` = -1 | **Human/domain rule** |
| Inconsistent labels | `Male` / `MALE` / `male` / `M` | **Human/business rule** |
| Duplicate customers | Repeated rows (same `Customer_ID`) | **Human/data governance rule** |
| ID variable | `Customer_ID` | **Human modeling decision** |
| Invalid/mixed dates | `31/02/2023`, `21-Mar-2021`, `07/29/2022` | **Human validation rule** |
| Target leakage | `Retention_Offer_Accepted_After_Lapse` | **Human/domain rule — must be removed** |
| Business interpretation | Is a ₹33+ lakh claim an error or a genuine large claim? | **Human judgment**, then leave for statistical treatment

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_FILE = "../datasets/insurance.csv"
df = pd.read_csv(DATA_FILE)

print("Shape:", df.shape)
df.head()


## Phase 1 — Inspect the raw dirty data before AutoML

A common mistake is to call AutoML immediately. First, we perform basic data understanding.

These checks are **not model preprocessing**; they are quality and meaning checks.

In [ ]:
print("\nDATA TYPES")
print(df.dtypes)

print("\nMISSING VALUES")
print(df.isna().sum().sort_values(ascending=False))

print("\nEXACT DUPLICATE ROWS:", df.duplicated().sum())

print("\nTARGET DISTRIBUTION")
print(df["Policy_Renewed"].value_counts(dropna=False))
print(df["Policy_Renewed"].value_counts(normalize=True, dropna=False).round(3))


In [ ]:
# Look at suspicious ranges and category spellings

print("AGE RANGE           :", df["Age"].min(), "to", df["Age"].max())
print("ANNUAL INCOME RANGE  :", df["Annual_Income"].min(), "to", df["Annual_Income"].max())
print("ANNUAL PREMIUM RANGE :", df["Annual_Premium"].min(), "to", df["Annual_Premium"].max())
print("TOTAL CLAIM RANGE    :", df["Total_Claim_Amount"].min(), "to", df["Total_Claim_Amount"].max())

for col in ["Gender", "Occupation", "Region", "Policy_Type", "Payment_Mode"]:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False))

print("\nSample of raw Policy_Start_Date values that will not parse as ISO dates:")
bad_dates = df.loc[pd.to_datetime(df["Policy_Start_Date"], errors="coerce").isna(), "Policy_Start_Date"]
print(bad_dates.tolist())


# Phase 2 — Human intervention BEFORE AutoML

The following corrections require knowledge of what the fields mean.

PyCaret cannot know that:
- a customer's age should fall within a sensible range for an active policyholder (we use 18-90);
- `9,999,999` is a sentinel/bad income value and `-1` is a sentinel/bad premium value in this dataset;
- `M`, `MALE`, and `male` all represent the same business category, and `W` is short for `West`;
- some `Policy_Start_Date` values are recorded in a different format (`21-Mar-2021`, `07/29/2022`)
  and at least one (`31/02/2023`) is not a real calendar date at all;
- `Retention_Offer_Accepted_After_Lapse` is only meaningful **after** a policy has already lapsed —
  it happens after the renewal outcome is known, so it leaks the answer.

We will correct these issues first, while deliberately leaving ordinary missing values, scaling,
encoding, outlier handling and class balancing for PyCaret.

In [ ]:
clean = df.copy()

# 1. Remove exact duplicate rows (repeated customer records).
before = len(clean)
clean = clean.drop_duplicates().copy()
print("Duplicates removed:", before - len(clean))

# 2. Convert impossible ages to missing.
#    A policyholder cannot sensibly be a small child or 145 years old.
#    IMPORTANT: We do NOT impute them manually; PyCaret will do the imputation later.
clean.loc[(clean["Age"] < 18) | (clean["Age"] > 90), "Age"] = np.nan

# 3. Convert known sentinel codes to missing.
#    9,999,999 is a placeholder used when income was not actually captured.
#    A negative/zero annual premium is not a real premium; it is an entry error.
clean.loc[clean["Annual_Income"] == 9999999, "Annual_Income"] = np.nan
clean.loc[clean["Annual_Premium"] <= 0, "Annual_Premium"] = np.nan

# 4. Standardize categorical labels using business rules.
clean["Gender"] = (
    clean["Gender"]
    .str.strip()
    .str.lower()
    .replace({"m": "Male", "male": "Male", "f": "Female", "female": "Female"})
)

clean["Occupation"] = (
    clean["Occupation"]
    .str.strip()
    .str.lower()
    .replace({
        "business": "Business",
        "salaried": "Salaried",
        "salary": "Salaried",
        "self employed": "Self-employed",
        "self-employed": "Self-employed",
        "retired": "Retired",
        "ret.": "Retired",
    })
)

clean["Region"] = (
    clean["Region"]
    .str.strip()
    .str.lower()
    .replace({"west": "West", "w": "West", "east": "East", "south": "South", "north": "North"})
)

clean["Payment_Mode"] = (
    clean["Payment_Mode"]
    .str.strip()
    .str.lower()
    .replace({
        "credit card": "Credit Card",
        "card": "Credit Card",
        "auto-debit": "Auto-debit",
        "auto debit": "Auto-debit",
        "upi": "UPI",
        "net banking": "Net Banking",
        "netbanking": "Net Banking",
    })
)

# IMPORTANT FOR PYCARET 3.x:
# pandas StringDtype uses pd.NA, which can cause
# "TypeError: boolean value of NA is ambiguous" inside scikit-learn's SimpleImputer.
# Convert categorical columns to ordinary Python object dtype and use np.nan.
categorical_cols = ["Gender", "Occupation", "Region", "Policy_Type", "Payment_Mode"]
for col in categorical_cols:
    clean[col] = clean[col].astype(object)
    clean[col] = clean[col].where(pd.notna(clean[col]), np.nan)

# 5. Parse the date using a HUMAN validation rule.
#    Most dates are ISO (YYYY-MM-DD), but a handful use DD/MM/YYYY, DD-Mon-YYYY,
#    or even MM/DD/YYYY (e.g. 07/29/2022, where day=29 proves it cannot be day-first).
#    A single fixed parsing rule cannot resolve every row correctly -- that judgment
#    call is exactly the kind of thing AutoML should not be trusted to make silently.
import warnings

def parse_date_mixed(value):
    text = str(value).strip()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        parsed = pd.to_datetime(text, errors="coerce", dayfirst=True)
        if pd.isna(parsed):
            # Fall back to month-first for the rare US-style entries.
            parsed = pd.to_datetime(text, errors="coerce", dayfirst=False)
    return parsed

clean["Policy_Start_Date"] = clean["Policy_Start_Date"].apply(parse_date_mixed)
print("Unparseable/impossible dates left as missing:", clean["Policy_Start_Date"].isna().sum())

# IMPORTANT ROBUSTNESS FIX:
# Rather than asking PyCaret to expand a raw date column that still contains NaT,
# derive a meaningful NUMERIC feature from it. `Policy_Tenure_Years` already exists
# as the business's own tenure counter, so we derive a complementary feature instead:
# how many days ago the policy started, using a fixed snapshot date for reproducibility.
SNAPSHOT_DATE = pd.Timestamp("2026-08-24")
clean["Days_Since_Policy_Start"] = (SNAPSHOT_DATE - clean["Policy_Start_Date"]).dt.days.astype(float)

# 6. Target leakage: DROP the post-outcome variable.
#    A retention offer is only extended after a policy has already lapsed, so its
#    value is not known at the moment we would actually need a renewal prediction.
clean = clean.drop(columns=["Retention_Offer_Accepted_After_Lapse"])

print("Shape after human intervention:", clean.shape)
clean.head()


In [ ]:
# Re-check the data after SEMANTIC cleaning.
# Notice: ordinary missing values are intentionally still present -- PyCaret will impute them.

print("\nMISSING VALUES LEFT FOR PYCARET")
print(clean.isna().sum().sort_values(ascending=False))

for col in ["Gender", "Occupation", "Region", "Policy_Type", "Payment_Mode"]:
    print(f"\n{col}:")
    print(clean[col].value_counts(dropna=False))

print("\nTarget distribution:")
print(clean["Policy_Renewed"].value_counts())
print(clean["Policy_Renewed"].value_counts(normalize=True).round(3))


# Phase 3 — Let PyCaret perform automated preprocessing

We now intentionally ask PyCaret to automate several preprocessing activities:

- numeric missing-value imputation using the median;
- categorical missing-value imputation using the mode;
- categorical encoding;
- z-score normalization;
- statistical outlier removal using Isolation Forest (this is *statistical*, not semantic,
  outlier handling — it will not know that a large claim can be a genuine high-value customer,
  but it is a reasonable automated first pass once the semantic issues above are already fixed);
- target imbalance treatment using SMOTE;
- train/test split and stratification;
- repeated model comparison through cross-validation.

The PyCaret 3.x `setup()` API documents these preprocessing controls. In recent PyCaret releases,
APIs may evolve, so verify the installed version when teaching.

In [ ]:
# OPTIONAL INSTALLATION
# Run this cell once if PyCaret is not installed.
# In Google Colab / Jupyter, restart the kernel after installation if requested.

#%pip install -U pycaret


In [ ]:
# ENVIRONMENT DIAGNOSTIC -- run this before PyCaret setup

import sys
import pandas as pd
import sklearn
import pycaret

print("Python       :", sys.version.split()[0])
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("PyCaret      :", pycaret.__version__)

print("\nCategorical dtypes before setup:")
print(clean[["Gender", "Occupation", "Region", "Policy_Type", "Payment_Mode"]].dtypes)

print("\nMissing-value Python types:")
for col in ["Occupation", "Region", "Payment_Mode"]:
    vals = clean.loc[clean[col].isna(), col]
    if len(vals):
        print(col, type(vals.iloc[0]), repr(vals.iloc[0]))

print("\nIf the missing values above show <class 'float'> nan,")
print("the pd.NA ambiguity problem has been removed.")


In [ ]:
# FINAL CHECK BEFORE PYCARET
print("Missing values before PyCaret (these should be ordinary np.nan):")
print(clean.isna().sum()[clean.isna().sum() > 0])

print("\nFeature dtypes:")
print(clean.dtypes)

import pycaret
print("PyCaret version:", pycaret.__version__)

from pycaret.classification import ClassificationExperiment

exp = ClassificationExperiment()

exp.setup(
    data=clean,
    target="Policy_Renewed",

    # Customer_ID is an identifier, not a predictive business feature.
    # Policy_Start_Date has already been converted into Days_Since_Policy_Start,
    # so the raw timestamp is no longer needed as an input.
    ignore_features=["Customer_ID", "Policy_Start_Date"],

    # Automated missing-value treatment
    imputation_type="simple",
    numeric_imputation="median",
    categorical_imputation="mode",

    # Automated encoding + scaling
    normalize=True,
    normalize_method="zscore",

    # Automated statistical outlier treatment
    remove_outliers=True,
    outliers_method="iforest",
    outliers_threshold=0.03,

    # Automated class balancing (renewals heavily outnumber non-renewals)
    fix_imbalance=True,

    # Evaluation design
    train_size=0.75,
    fold=5,
    data_split_stratify=True,
    session_id=123,
    verbose=True,
)


In [ ]:
# VERIFY THAT SETUP COMPLETED SUCCESSFULLY

print("PyCaret setup completed.")
print("Training rows:", len(exp.get_config("X_train")))
print("Test rows    :", len(exp.get_config("X_test")))
print("\nPipeline:")
print(exp.get_config("pipeline"))


## Inspect the preprocessing pipeline

Rather than treating AutoML as a black box, inspect the pipeline.

This is an especially useful classroom step: students can see that AutoML constructs a sequence
of transformations before the estimator.

In [ ]:
pipeline = exp.get_config("pipeline")
print(pipeline)


### Five (of the) automated preprocessing operations, explained

1. **Numeric missing-value imputation (median)** — `Age`, `Annual_Income`, `Annual_Premium` and
   `Days_Since_Policy_Start` each still contain ordinary missing values after our semantic
   cleaning. PyCaret's `SimpleImputer` fills these with the column median, which is robust to
   the skew typical of income and premium data.
2. **Categorical missing-value imputation (mode)** — `Occupation`, `Region` and `Payment_Mode`
   still have a handful of missing entries. PyCaret fills these with the most frequent category
   for that column.
3. **Categorical encoding** — `Gender`, `Occupation`, `Region`, `Policy_Type` and `Payment_Mode`
   are converted from text labels into numeric representations the estimator can use, without us
   writing any manual `map`/`get_dummies` code.
4. **Z-score normalization** — numeric columns on very different scales (income in lakhs vs.
   number of claims in single digits) are rescaled to comparable ranges so that no feature
   dominates purely because of its units.
5. **Statistical outlier removal (Isolation Forest)** — after the sentinel/impossible values were
   already handled by us, PyCaret flags and removes a small proportion (`outliers_threshold=0.03`)
   of remaining statistically unusual training rows.
6. **Class imbalance treatment (SMOTE)** — because far more customers renew than not, PyCaret
   synthesizes minority-class (`No`) examples during training so the estimator does not simply
   learn to predict "Yes" every time.
7. **Stratified train/test split with cross-validation** — `train_size=0.75` and `fold=5` together
   ensure the target's Yes/No proportions are preserved in every split and that model comparison
   in the next phase is not based on a single lucky/unlucky split.

In [ ]:
# Inspect transformed data using PyCaret configuration.
# This is safer than directly forcing every sampler/transformer to transform.

try:
    X_train_transformed = exp.get_config("X_train_transformed")
    y_train_transformed = exp.get_config("y_train_transformed")

    X_train = exp.get_config("X_train")
    print("Raw training feature shape        :", X_train.shape)
    print("Transformed training feature shape:", X_train_transformed.shape)
    print("\nMissing values after preprocessing:",
          int(X_train_transformed.isna().sum().sum()))
    display(X_train_transformed.head())
except Exception as e:
    print("Your PyCaret version does not expose X_train_transformed in this way.")
    print("The pipeline itself can still be inspected with exp.get_config('pipeline').")
    print("Message:", e)


# Phase 4 — AutoML model comparison

`compare_models()` evaluates multiple classification algorithms using cross-validation and ranks
them using the chosen metric.

Renewals heavily outnumber non-renewals in this dataset, so **AUC or F1** are more informative
than accuracy alone — a model that always predicts "Yes" would already score a high accuracy
without being useful for identifying the customers actually at risk of not renewing.

In [ ]:
best_model = exp.compare_models(sort="AUC")
best_model


In [ ]:
leaderboard = exp.pull()
leaderboard.head(10)


## Evaluate the selected model on the hold-out set

In [ ]:
predictions = exp.predict_model(best_model)
predictions.head()


In [ ]:
# Common evaluation plots.
# Run individually if a plotting backend/version raises an issue.

exp.plot_model(best_model, plot="confusion_matrix")


In [ ]:
exp.plot_model(best_model, plot="auc")


In [ ]:
exp.plot_model(best_model, plot="feature")


# Wrap-up — connecting back to the FinServe case

`reference_documents/automl-data-preparation-case-finserve.docx` asks three discussion questions
about a lending model. The same reasoning applies directly to what we just did with the insurance
renewal model:

**1. Which data-preparation activities belong to AutoML vs. a human?** Ordinary missing values,
scaling, categorical encoding, statistical outlier removal and class imbalance were safely left to
PyCaret — these are mechanical, repeatable operations. Impossible ages, sentinel income/premium
codes, inconsistent category spellings, exact duplicates, the `Customer_ID` identifier, invalid or
mixed-format dates, and — most importantly — the `Retention_Offer_Accepted_After_Lapse` leakage
variable all required a human who understood what the fields *mean* in an insurance business
context. PyCaret has no way to know that a retention offer is only made after a policy has
already lapsed.

**2. Why might a higher headline accuracy not mean a better model?** If we had left
`Retention_Offer_Accepted_After_Lapse` in the training data, the model would have looked
extremely accurate — almost every customer who was offered (and refused) a retention deal did
not renew — but that variable would not exist at the moment we actually need the prediction
(before the policy has lapsed). A model "boosted" by leakage cannot be deployed for its intended
purpose: it is answering a question we can no longer ask in production. The same logic applied to
`Collection_Status` in the FinServe case.

**3. Recommendation.** As with FinServe, the right choice is the **human-supervised AutoML
model**: remove the leakage variable and the other semantic issues first, then let PyCaret
automate the rest and compare models using AUC/F1 rather than accuracy alone. Two controls worth
adding for future AutoML use: (a) a mandatory human data-review checklist — impossible values,
sentinel codes, identifiers, leakage variables — signed off *before* any AutoML run, and (b) a
standing rule that any feature only defined after the prediction moment (`*_After_*`,
`*_Post_*`, anything generated by a downstream business process) is excluded from `setup()` by
default and must be explicitly justified to be included.